In [ ]:
!pip -q install -U transformers accelerate peft wandb pyarrow
!pip -q uninstall -y torchao || true          # peft needs torchao>=0.16 OR none
import os, gc, numpy as np, pandas as pd, torch
print("CUDA:", torch.cuda.is_available())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/data/taxi_series.parquet"
FINAL_DIR = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/timesfm_ft_final"
RESOLUTIONS = []        # [] = all 9 resolutions (production)
os.makedirs(FINAL_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
if RESOLUTIONS:
    df = df[df.resolution.isin(RESOLUTIONS)].copy()
df["ts"] = pd.to_datetime(df["ts"]); df = df.sort_values(["series_id","split","ts"])
def to_dict(sp): return {s: g.sort_values("ts")["value"].to_numpy(np.float32)
                         for s, g in df[df.split==sp].groupby("series_id")}
TRAIN, VAL = to_dict("train"), to_dict("val")
META = df[["series_id","metric","resolution","vendor"]].drop_duplicates().set_index("series_id")
FULL = {s: np.concatenate([TRAIN[s], VAL.get(s, np.array([],np.float32))]) for s in TRAIN}
VAL_START = {s: len(TRAIN[s]) for s in TRAIN}
print("series:", len(TRAIN))


In [ ]:
CONTEXT, HORIZON, EVAL_MAX_WINDOWS = 512, 24, 150
MODEL_ID = "google/timesfm-2.5-200m-transformers"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from transformers import TimesFm2_5ModelForPrediction

def load_base():
    return TimesFm2_5ModelForPrediction.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16 if DEVICE=="cuda" else torch.float32,
        device_map=DEVICE)

def wape(y,p):
    y,p=np.asarray(y,float),np.asarray(p,float); d=np.abs(y).sum()
    return float(np.abs(y-p).sum()/d*100) if d else float("nan")

class RandomWindowDS(torch.utils.data.Dataset):
    def __init__(self, series, C, H, n, seed=42):
        self.s, self.C, self.H = series, C, H
        rng=np.random.default_rng(seed); mn=C+H
        valid=[i for i,x in enumerate(series) if len(x)>=mn]; self.items=[]
        for _ in range(n):
            i=int(rng.choice(valid)); st=int(rng.integers(0,len(series[i])-mn+1)); self.items.append((i,st))
    def __len__(self): return len(self.items)
    def __getitem__(self,k):
        i,st=self.items[k]; x=self.s[i]
        return (torch.tensor(x[st:st+self.C],dtype=torch.float32),
                torch.tensor(x[st+self.C:st+self.C+self.H],dtype=torch.float32))

@torch.no_grad()
def eval_wape_per_res(model, batch=128):
    windows=[]
    for sid,arr in FULL.items():
        vs=VAL_START[sid]; origins=[o for o in range(vs,len(arr)-HORIZON+1,HORIZON) if o-CONTEXT>=0]
        if len(origins)>EVAL_MAX_WINDOWS:
            origins=[origins[i] for i in np.linspace(0,len(origins)-1,EVAL_MAX_WINDOWS).astype(int)]
        for o in origins: windows.append((sid,arr[o-CONTEXT:o],arr[o:o+HORIZON]))
    per={}
    for i in range(0,len(windows),batch):
        ch=windows[i:i+batch]
        X=torch.tensor(np.stack([c for _,c,_ in ch]),dtype=torch.float32,device=DEVICE)
        mp=model(past_values=X).mean_predictions[:,:HORIZON].float().cpu().numpy()
        for j,(sid,_,tgt) in enumerate(ch):
            per.setdefault(sid,([],[])); per[sid][0].append(tgt); per[sid][1].append(mp[j])
    per_res={}
    for sid,(ys,ps) in per.items():
        w=wape(np.concatenate(ys),np.concatenate(ps)); r=META.loc[sid].resolution
        per_res.setdefault(r,[]).append(w)
    return per_res


In [ ]:
from peft import LoraConfig, get_peft_model

# ▼▼▼ REPLACE with your TimesFM Optuna study.best_params ▼▼▼
BEST_PARAMS = {"lr": 5.448439034596534e-05, "lora_r": 4, "lora_alpha": 32,
               "weight_decay": 0.08093498333840617}
# ▲▲▲ ------------------------------------------------- ▲▲▲

EPOCHS, BATCH, NUM_SAMPLES = 3, 16, 12000     # all-res → more windows than the 1h HPO run

TRAIN_LIST = list(TRAIN.values())
model = get_peft_model(load_base(), LoraConfig(
    r=BEST_PARAMS["lora_r"], lora_alpha=BEST_PARAMS["lora_alpha"],
    target_modules="all-linear", lora_dropout=0.05, bias="none")).train()

ds = RandomWindowDS(TRAIN_LIST, CONTEXT, HORIZON, NUM_SAMPLES)
dl = torch.utils.data.DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)
opt = torch.optim.AdamW(model.parameters(), lr=BEST_PARAMS["lr"],
                        weight_decay=BEST_PARAMS["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS*len(dl))

for ep in range(1, EPOCHS+1):
    model.train()
    for ctx, tgt in dl:
        ctx, tgt = ctx.to(DEVICE), tgt.to(DEVICE)
        out = model(past_values=ctx, future_values=tgt, forecast_context_len=CONTEXT)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); opt.zero_grad(); sched.step()
    model.eval()
    per_res = eval_wape_per_res(model)
    allv = [w for ws in per_res.values() for w in ws]
    print(f"epoch {ep} · overall median WAPE: {np.nanmedian(allv):.3f}")

# final per-resolution report
per_res = eval_wape_per_res(model)
print("=== FINAL all-res holdout WAPE (per resolution) ===")
for r in sorted(per_res): print(f"  {r:>4}: {np.nanmedian(per_res[r]):.3f}")
allv = [w for ws in per_res.values() for w in ws]
print("overall median WAPE:", round(float(np.nanmedian(allv)), 3))

model.save_pretrained(FINAL_DIR)
print("saved adapter →", FINAL_DIR)
